# UNSW-NB15 preprocessing and leakage audit — V03 actual run

The raw CSV was preserved. No imputation, clipping or silent duplicate deletion was used. This notebook reads the completed V03 reports and shows the exact before/after and leakage evidence.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
sys.path.insert(0, str(root / 'src'))
output = root / 'artifacts/experiments/unsw_v03_research_completeness_actual'
report = json.loads((output / 'preprocessing_report.json').read_text(encoding='utf-8'))
display(pd.DataFrame([report]))
display(pd.read_csv(output / 'class_distribution_before_after.csv'))
display(pd.read_csv(output / 'quantiles_and_outliers.csv'))

,status,raw_rows,processed_rows,removed_rows,retention_rate,removal_reasons,raw_duplicate_rows_reported_not_removed,processed_duplicate_feature_rows_reported_not_removed,quality_audit,policy
0,COMPLETED,175341,143229,32112,0.81686,"{'unsupported_protocol': 32112, 'negative_or_n...",0,68352,"{'nan_counts_by_physical_column': {'dur': 0, '...","{'raw_preserved': True, 'imputation': 'none', ..."


,stage,label,rows
0,RAW_BEFORE,1,119341
1,RAW_BEFORE,0,56000
2,PROCESSED_AFTER,1,90186
3,PROCESSED_AFTER,0,53043


,feature,q01,q05,q25,q50,q75,q95,q99,iqr_lower_fence,iqr_upper_fence,iqr_outlier_rows
0,duration_s,0.000001,0.000003,0.000009,0.028998,0.772624,2.709821,18.20202,-1.158913,1.931546,10929
1,packets,2.000000,2.000000,2.000000,16.000000,32.000000,138.000000,674.00000,-43.000000,77.000000,18354
2,bytes,104.000000,114.000000,114.000000,1024.000000,4946.000000,67754.000000,561774.00000,-7134.000000,12194.000000,23209


In [2]:
samples = pd.read_csv(output / 'before_after_samples.csv')
before = samples.loc[samples['stage'] == 'RAW_BEFORE', 'source_raw_row'].tolist()
after = samples.loc[samples['stage'] == 'PROCESSED_AFTER', 'source_raw_row'].tolist()
assert before == after
display(samples)
print('Matching source rows:', before)

,stage,source_raw_row,id,dur,proto,service,state,spkts,dpkts,sbytes,...,is_sm_ips_ports,attack_cat,label,duration_s,packets,bytes,protocol_tcp,duration_log,packets_log,bytes_log
0,RAW_BEFORE,0,1,0.121478,tcp,-,FIN,6,4,258,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RAW_BEFORE,1,2,0.649902,tcp,-,FIN,14,38,734,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RAW_BEFORE,2,3,1.623129,tcp,-,FIN,8,16,364,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RAW_BEFORE,3,4,1.681642,tcp,ftp,FIN,12,12,628,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RAW_BEFORE,4,5,0.449454,tcp,-,FIN,10,6,534,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RAW_BEFORE,5,6,0.380537,tcp,-,FIN,10,6,534,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RAW_BEFORE,6,7,0.637109,tcp,-,FIN,10,8,534,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,RAW_BEFORE,7,8,0.521584,tcp,-,FIN,10,8,534,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,RAW_BEFORE,8,9,0.542905,tcp,-,FIN,10,8,534,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,RAW_BEFORE,9,10,0.258687,tcp,-,FIN,10,6,534,...,0,Normal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Matching source rows: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [3]:
leakage = json.loads((output / 'leakage_audit.json').read_text(encoding='utf-8'))
display(leakage)
print('Official Test accessed:', leakage['official_test_accessed'])

{'status': 'COMPLETED',
 'official_test_accessed': False,
 'official_test_path': 'data\\raw\\unsw-nb15\\UNSW_NB15_testing-set.csv',
 'label_columns_excluded_from_model': ['label', 'attack_cat'],
 'model_features': ['protocol_tcp',
  'duration_log',
  'packets_log',
  'bytes_log'],
 'direct_label_feature_overlap': [],
 'exact_duplicate_raw_rows': 0,
 'exact_duplicate_feature_rows': 68352,
 'feature_groups_with_conflicting_binary_labels': 282,
 'partition_feature_group_overlap': {'development_vs_validation': 0,
  'calibration_vs_development': 0,
  'calibration_vs_validation': 0},
 'cv_group_definition': 'Exact equality of all four transformed v02 feature values',
 'host_time_leakage': {'available_columns': [],
  'status': 'LIMITED_NOT_AVAILABLE',
  'limitation': 'UNSW-NB15 training contract does not expose a verified host/session/time identity; feature-disjoint is not host/time-disjoint.'},
 'official_test_policy': 'Frozen official test is not opened, fitted, tuned or scored by v03.'}

Official Test accessed: False
